# Aspect + top-10-reasons — ASTE (t5-base) on SemEval Triplet restaurant data

Runs the fine-tuned **t5-base** ASTE model (`notebooks/train-t5-base-for-aste-on-14res-15res-16res.ipynb`)
over the union of `train`/`dev`/`test` (14res+15res+16res), then aggregates the resulting
`(aspect, opinion, sentiment)` triplets into **one row per aspect** with `positive`/`negative`/
`neutral` counts *and* the top-10 most frequent opinion phrases ("reasons") for each sentiment —
the input the FLAN-T5 report-generation step consumes.

Both a **gold** table (from the dataset's own triplet annotations) and a **predicted** table
(from the model) are produced, and compared as a sanity check before handing the predicted table
off — the same pattern already used for the Laptop/BERT track in
`notebooks/aspect_stats_semeval_laptop.ipynb`.

**Why on Kaggle, not local**: t5-base beam-search generation over ~4500 sentences takes ~3+ hours
on CPU; on a Kaggle GPU it's minutes. The aggregation logic itself
(`src/report/aspect_stats.py::aggregate_aspect_reasons`, inlined below) is plain Python and is
unit-tested locally (`tests/test_aspect_stats.py`) — only the model inference needs a GPU.

**Kaggle setup**: add `dattm03/genai-dataset` is not needed here (different dataset/domain) —
instead add the t5-base ASTE model as a Kaggle Dataset/Model (upload
`t5-base-aste-restaurant-best/`, saved by the training notebook's Output tab), framework
**PyTorch**. GPU accelerator + internet (for the dataset git-clone fallback, unless you've also
uploaded `semi-triple-{14,15,16}res` as inputs).

In [ ]:
import glob
import json
import re
import subprocess
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


## 1. Locate the data and the fine-tuned model

Same `DOMAINS`/`find_domain_dir`/`find_split_file`/`collect_dataset_files` cell as the ASTE
training notebooks (`notebooks/train-t5-base-for-aste-on-14res-15res-16res.ipynb`), reused
verbatim. `find_model_dir` matches on `aste-restaurant-best` — the exact suffix
`BEST_MODEL_DIR` is saved under — to avoid picking a `*-aste-restaurant` training checkpoint
or a `*-aste-seed<N>*` dir if the Kaggle Model/Dataset was created via "New Model from notebook
output" (same lesson learned from the BERT/DistilBERT episode earlier in this project).

In [ ]:
DOMAINS = ["14res", "15res", "16res"]
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
KAGGLE_DOMAIN_DIRS = {
    "14res": INPUT_ROOT / "semi-triple-14res",
    "15res": INPUT_ROOT / "semi-triple-15res",
    "16res": INPUT_ROOT / "semi-triple-16res",
}


def find_domain_dir(root: Path, domain: str):
    if not root.exists():
        return None
    candidates = []
    explicit_dir = KAGGLE_DOMAIN_DIRS.get(domain)
    if explicit_dir is not None and explicit_dir.exists() and any(explicit_dir.glob("*.txt")):
        candidates.append(explicit_dir)
    for p in root.rglob("*"):
        if p.is_dir() and domain in p.name.lower() and any(p.glob("*.txt")):
            candidates.append(p)
    return sorted(candidates)[0] if candidates else None


def find_split_file(domain_dir: Path, split: str):
    files = sorted(domain_dir.glob("*.txt"))
    names = [f.name.lower() for f in files]

    if split == "train":
        preferred = ["train.txt", f"{domain_dir.name}_train.txt", f"{domain_dir.name}t_train.txt", f"{domain_dir.name}rest_train.txt"]
        patterns = ["train"]
    elif split == "dev":
        preferred = ["dev.txt", "val.txt", f"{domain_dir.name}_dev.txt", f"{domain_dir.name}t_dev.txt", f"{domain_dir.name}rest_dev.txt"]
        patterns = ["dev", "val"]
    elif split == "test":
        preferred = ["test.txt", f"{domain_dir.name}_test.txt", f"{domain_dir.name}t_test.txt", f"{domain_dir.name}rest_test.txt"]
        patterns = ["test"]
    else:
        raise ValueError(split)

    for name in preferred:
        for f in files:
            if f.name.lower() == name.lower():
                return f
    for f, name in zip(files, names):
        if any(pat in name for pat in patterns):
            return f
    return None


def collect_dataset_files(root: Path):
    found = {}
    for domain in DOMAINS:
        d = find_domain_dir(root, domain)
        if d is None:
            continue
        split_files = {split: find_split_file(d, split) for split in ["train", "dev", "test"]}
        if all(split_files.values()):
            found[domain] = split_files
    return found


dataset_files = collect_dataset_files(INPUT_ROOT)
if len(dataset_files) < 3:
    repo_dir = WORKING_ROOT / "SemEval-Triplet-data"
    if not repo_dir.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/xuuuluuu/SemEval-Triplet-data.git",
            str(repo_dir),
        ])
    dataset_files = collect_dataset_files(repo_dir)

missing_domains = [d for d in DOMAINS if d not in dataset_files]
assert not missing_domains, f"Missing domains: {missing_domains}. Add dataset to Kaggle Input or enable Internet."
print(json.dumps({d: {s: str(p) for s, p in splits.items()} for d, splits in dataset_files.items()}, indent=2))


def find_model_dir():
    candidates = []
    for config_path in glob.glob(f"{INPUT_ROOT}/**/config.json", recursive=True):
        d = Path(config_path).parent
        if (d / "model.safetensors").exists() or (d / "pytorch_model.bin").exists():
            candidates.append(d)
    candidates = sorted(set(candidates), key=str)
    final_candidates = [d for d in candidates if "aste-restaurant-best" in str(d).lower()]
    chosen = final_candidates or candidates
    if not chosen:
        raise FileNotFoundError(
            "No fine-tuned model found under /kaggle/input. Upload the t5-base-aste-restaurant-best/ "
            "folder saved by notebooks/train-t5-base-for-aste-on-14res-15res-16res.ipynb as a new "
            "Kaggle Dataset/Model and add it as this notebook's input."
        )
    if len(candidates) > 1:
        print(f"Found {len(candidates)} candidate model dir(s): {candidates}")
    return chosen[0]


MODEL_DIR = find_model_dir()
print("Using model dir:", MODEL_DIR)


## 2. Parse ASTE triplet format

Same logic as `src/data/aste_loader.py` (tested locally in `tests/test_aste_loader.py`), inlined
here so the notebook is self-contained on Kaggle.

In [ ]:
SENTIMENT_MAP = {"POS": "positive", "NEG": "negative", "NEU": "neutral"}


@dataclass
class AsteTriplet:
    aspect: str
    opinion: str
    sentiment: str


@dataclass
class AsteSentence:
    text: str
    triplets: list = field(default_factory=list)


def _split_token_tag(item):
    token, tag = item.rsplit("=", 1)
    return token, tag


def _parse_tag_sequence(tag_text):
    return [_split_token_tag(item) for item in tag_text.strip().split()]


def _phrase_from_tokens(tokens):
    return " ".join(tokens).replace(" n\'t", "n\'t").replace(" \'s", "\'s").strip()


def parse_aste_line(line):
    parts = line.strip().split("####")
    if len(parts) != 3:
        return None

    sentence, target_tag_text, opinion_tag_text = parts
    target_pairs = _parse_tag_sequence(target_tag_text)
    opinion_pairs = _parse_tag_sequence(opinion_tag_text)

    target_groups = {}
    for token, tag in target_pairs:
        if tag == "O" or "-" not in tag:
            continue
        group_id, sentiment_code = tag.split("-", 1)
        target_groups.setdefault(group_id, {"tokens": [], "sentiment": sentiment_code})
        target_groups[group_id]["tokens"].append(token)

    opinion_groups = {}
    for token, tag in opinion_pairs:
        if tag == "O":
            continue
        opinion_groups.setdefault(tag, []).append(token)

    triplets = []
    for group_id, target_info in sorted(target_groups.items(), key=lambda x: (len(x[0]), x[0])):
        opinion_group_id = "S" * len(group_id)
        aspect = _phrase_from_tokens(target_info["tokens"])
        opinion = _phrase_from_tokens(opinion_groups.get(opinion_group_id, []))
        sentiment = SENTIMENT_MAP.get(target_info["sentiment"], target_info["sentiment"].lower())
        if aspect and opinion:
            triplets.append(AsteTriplet(aspect=aspect, opinion=opinion, sentiment=sentiment))

    return AsteSentence(text=sentence.strip(), triplets=triplets)


def load_aste_file(path):
    sentences = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = parse_aste_line(line)
            if item is not None:
                sentences.append(item)
    return sentences


all_sentences = []
for domain, splits in dataset_files.items():
    for split, path in splits.items():
        all_sentences.extend(load_aste_file(path))

print(f"Total sentences (train+dev+test, all domains): {len(all_sentences)}")


## 3. Run t5-base inference over every sentence

Batched beam-search generation on GPU, same `PREFIX` the model was fine-tuned with.

In [ ]:
PREFIX = "extract aspect sentiment triplets: "
MAX_LENGTH = 160
BATCH_SIZE = 32
TRIPLET_RE = re.compile(
    r"aspect:\s*(.*?)\s*\|\s*opinion:\s*(.*?)\s*\|\s*sentiment:\s*(positive|negative|neutral)",
    re.IGNORECASE,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print("Device:", device)


def parse_predicted_triplets(text):
    """Unlike the training notebook's metric-only parser (which dedupes into a `set` for P/R/F1
    scoring), every individual triplet is kept here since the aggregation counts mentions."""
    if text.strip().lower() == "no triplet":
        return []
    return [
        (aspect.strip(), opinion.strip(), sentiment.strip().lower())
        for aspect, opinion, sentiment in TRIPLET_RE.findall(text)
    ]


texts = [s.text for s in all_sentences]
predicted_texts = []
with torch.no_grad():
    for start in range(0, len(texts), BATCH_SIZE):
        batch = [PREFIX + t for t in texts[start : start + BATCH_SIZE]]
        inputs = tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        output_ids = model.generate(**inputs, max_length=MAX_LENGTH, num_beams=4, early_stopping=True)
        predicted_texts.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
        done = min(start + BATCH_SIZE, len(texts))
        print(f"  {done}/{len(texts)}", end="\r")
print()

predicted_records = [triplet for text in predicted_texts for triplet in parse_predicted_triplets(text)]
gold_records = [(t.aspect, t.opinion, t.sentiment) for s in all_sentences for t in s.triplets]
print(f"Gold triplets: {len(gold_records)} | Predicted triplets: {len(predicted_records)}")


## 4. Aggregate: per-aspect counts + top-10 reasons per sentiment

Same logic as `src/report/aspect_stats.py::aggregate_aspect_reasons` (tested locally in
`tests/test_aspect_stats.py`), inlined here. One row per aspect:
`{aspect, positive, positive_reasons, negative, negative_reasons, neutral, neutral_reasons,
total, majority_sentiment}` — `*_reasons` are the top-N most frequent opinion phrases for that
aspect+sentiment, as `[phrase, count]` pairs.

In [ ]:
SENTIMENT_LABELS = ("positive", "negative", "neutral")


@dataclass
class AspectReasonSummary:
    aspect: str
    positive: int
    positive_reasons: list
    negative: int
    negative_reasons: list
    neutral: int
    neutral_reasons: list
    total: int
    majority_sentiment: str


def _top_reasons(counter, top_n):
    ranked = sorted(counter.items(), key=lambda item: (-item[1], item[0]))
    return ranked[:top_n]


def aggregate_aspect_reasons(records, top_n=10, min_mentions=1):
    counts = {}
    reason_counts = {}
    for aspect, opinion, sentiment in records:
        key = aspect.strip().lower()
        if not key or sentiment not in SENTIMENT_LABELS:
            continue
        counts.setdefault(key, Counter())[sentiment] += 1
        reason = opinion.strip().lower()
        if reason:
            reason_counts.setdefault(key, {}).setdefault(sentiment, Counter())[reason] += 1

    summaries = []
    for aspect, counter in counts.items():
        total = sum(counter.values())
        if total < min_mentions:
            continue
        majority_sentiment = max(
            SENTIMENT_LABELS, key=lambda label: (counter[label], -SENTIMENT_LABELS.index(label))
        )
        aspect_reasons = reason_counts.get(aspect, {})
        summaries.append(
            AspectReasonSummary(
                aspect=aspect,
                positive=counter["positive"],
                positive_reasons=_top_reasons(aspect_reasons.get("positive", Counter()), top_n),
                negative=counter["negative"],
                negative_reasons=_top_reasons(aspect_reasons.get("negative", Counter()), top_n),
                neutral=counter["neutral"],
                neutral_reasons=_top_reasons(aspect_reasons.get("neutral", Counter()), top_n),
                total=total,
                majority_sentiment=majority_sentiment,
            )
        )

    summaries.sort(key=lambda s: (-s.total, s.aspect))
    return summaries


MIN_MENTIONS = 2
TOP_N = 10

gold_summary = aggregate_aspect_reasons(gold_records, top_n=TOP_N, min_mentions=MIN_MENTIONS)
predicted_summary = aggregate_aspect_reasons(predicted_records, top_n=TOP_N, min_mentions=MIN_MENTIONS)

print(f"Aspects with >= {MIN_MENTIONS} mentions: gold={len(gold_summary)}, predicted={len(predicted_summary)}")
print()
print("Top 10 aspects by mentions (predicted):")
for s in predicted_summary[:10]:
    print(f"  {s.aspect:<20} total={s.total:3d}  +{s.positive:<3d} -{s.negative:<3d} ~{s.neutral:<3d}  majority={s.majority_sentiment}")
    reasons_by_sentiment = {"positive": s.positive_reasons, "negative": s.negative_reasons, "neutral": s.neutral_reasons}
    top_reasons = reasons_by_sentiment[s.majority_sentiment]
    if top_reasons:
        print(f"    top reasons: {top_reasons[:3]}")


## 5. Sanity check: gold vs. predicted majority sentiment

In [ ]:
gold_by_aspect = {s.aspect: s for s in gold_summary}
predicted_by_aspect = {s.aspect: s for s in predicted_summary}
shared_aspects = set(gold_by_aspect) & set(predicted_by_aspect)

matches = sum(
    gold_by_aspect[a].majority_sentiment == predicted_by_aspect[a].majority_sentiment
    for a in shared_aspects
)
agreement = matches / len(shared_aspects) if shared_aspects else 0.0
print(f"Aspects in both tables: {len(shared_aspects)}")
print(f"Majority-sentiment agreement: {matches}/{len(shared_aspects)} = {agreement:.4f}")


## 6. Save results

In [ ]:
def to_records(summary):
    return [
        {
            "aspect": s.aspect,
            "positive": s.positive,
            "positive_reasons": s.positive_reasons,
            "negative": s.negative,
            "negative_reasons": s.negative_reasons,
            "neutral": s.neutral,
            "neutral_reasons": s.neutral_reasons,
            "total": s.total,
            "majority_sentiment": s.majority_sentiment,
        }
        for s in summary
    ]


out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("output")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "aspect_reasons_restaurant.json"
out_path.write_text(json.dumps({
    "model_dir": str(MODEL_DIR),
    "domain": "restaurant (14res+15res+16res)",
    "min_mentions": MIN_MENTIONS,
    "top_n_reasons": TOP_N,
    "num_examples": len(all_sentences),
    "majority_sentiment_agreement": agreement,
    "gold": to_records(gold_summary),
    "predicted": to_records(predicted_summary),
}, indent=2))
print(f"Saved aspect + reasons table to {out_path}")


## Next steps

- Hand off `aspect_reasons_restaurant.json` (the `predicted` table) to the FLAN-T5
  report-generation step — each row now carries not just aspect+sentiment counts but example
  quoted reasons per sentiment, ready to cite in generated report sentences.
- If `majority_sentiment_agreement` is notably lower than the model's held-out test triplet-F1
  (0.7442, see `plans/project-plan.md` Tuần 4), check which aspects flip — likely low-mention
  aspects near a count tie, which `MIN_MENTIONS` can be raised to filter out.
- `src/report/aspect_stats.py::aggregate_aspect_reasons` and `src/data/aste_loader.py` are the
  tested source of truth this notebook inlines from — if the aggregation logic needs to change,
  update those first (and their tests), then re-sync this notebook.